![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Week 2, Day 3 -- Lab 3: Image Search with a Frozen EfficientNet

A pretrained model can turn any image into a short list of numbers -- an
**embedding** -- that captures what's in it. Similar images get similar numbers.

We **freeze** EfficientNet (use it without training -- pure transfer learning),
turn a set of images into embeddings, and find look-alikes with **cosine
similarity**.

## 1) Setup

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
import torchvision
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

## 2) Load a frozen EfficientNet (no training)

We drop its classifier so the model outputs the **embedding** instead of a label.

In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)
model.classifier = torch.nn.Identity()   # keep the features, remove the classifier
model.eval();

## 3) Get a gallery of images

We use 200 images from CIFAR-10.

In [ ]:
prep = weights.transforms()              # the resize/normalise this model expects
gallery = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=prep)
shown   = torchvision.datasets.CIFAR10("./data", train=False)   # same images, for display
N = 200

## 4) Turn every image into an embedding

In [ ]:
vectors = []
with torch.no_grad():
    for i in range(0, N, 50):
        batch = torch.stack([gallery[j][0] for j in range(i, min(i + 50, N))])
        vectors.append(model(batch))
embeddings = torch.cat(vectors).numpy()
print("embeddings:", embeddings.shape)

## 5) Simplify with PCA

PCA squeezes each 1280-number embedding down to 50 -- faster and less noisy.

In [ ]:
reduced = PCA(n_components=50).fit_transform(embeddings)

## 6) Search for the 5 most similar images

Pick a query image; cosine similarity ranks every other image against it.

In [ ]:
query = 0                                       # try other numbers!
scores = cosine_similarity([reduced[query]], reduced)[0]
top5 = scores.argsort()[::-1][1:6]              # most similar (skip the query itself)

## 7) Show the query and its look-alikes

In [ ]:
plt.figure(figsize=(12, 3))
plt.subplot(1, 6, 1); plt.imshow(shown[query][0]); plt.title("query"); plt.axis("off")
for j, idx in enumerate(top5):
    plt.subplot(1, 6, j + 2); plt.imshow(shown[idx][0]); plt.title(f"#{j+1}"); plt.axis("off")
plt.show()

## Try it yourself

- Change `query` to a different number and re-run steps 6-7.
- The model was never trained on this task -- it just *understands* images well
  enough that similar ones land near each other. That's the power of transfer
  learning.

### *Contributed By: Sattam Altwaim*